# Scaffolding project

_DSAIT4050: Information retrieval lecture, TU Delft_

Welcome to the **DSAIT4050: Information retrieval** lecture!

This project acts as a gentle introduction to information retrieval for you. You do not need any prior knowledge about IR for this task. Only some Python programming skills are required.

## Getting started

Under the hood, this notebook uses a library called **PyTerrier**. Please check out the first part of our _Introduction to PyTerrier_ series to learn how to install PyTerrier. However, you do not need to interact with PyTerrier directly for now; rather, we're providing you with simple utility functions you can use. Feel free to have a look how these are implemented, but it's not required.

**Task 1**: Install PyTerrier (see the `01-setup.ipynb` notebook).

Now you should be able to import the utility functions. A dataset will be downloaded and indexed automatically (this will take a minute).


In [1]:
pip install python-terrier==0.12.1

Note: you may need to restart the kernel to use updated packages.


In [3]:
from util import search, evaluate, evaluate_all, get_tf_idf

Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
antique/test/non-offensive documents: 100%|██████████| 403666/403666 [00:00<00:00, 724284.33it/s]


Now that we have loaded the data, you can run search queries. For example:


In [2]:
search("what is the meaning of life")

NameError: name 'search' is not defined

What you get here is a list of ten documents from the corpus that are ordered by how relevant they are to our query (according to the search engine).

## Query rewriting

The goal of this task is to come up with a way of **rewriting queries** such that the search engine can "understand" them better.

In order to do this, let's first take a look at some example queries from our dataset. We represent these queries using a `pandas.DataFrame`, where the first column corresponds to the **query ID** and the second column corresponds to the **query**:


In [4]:
import pandas as pd

example_queries = pd.DataFrame(
    [
        [
            "443848",
            "does anybody know where i could get a free guide on how to train a siberian husky",
        ],
        [
            "1783010",
            "what is blaphsemy",
        ],
        [
            "2838988",
            "how can i get a cork out of not into a wine bottle without a corkscrew",
        ],
    ],
    columns=["qid", "query"],
)

Since these queries are taken from the dataset, we can **evaluate the performance** of our search engine on these queries. This means that we know which documents the system should retrieve for each query.

You can use the following evaluation function to do this. This function takes your queries and returns a score (mean average precision -- you will learn about this later). For now, all you need to know is that, the higher this score, the better the system works.

Let's evaluate the queries we have:


In [7]:
print("score:", evaluate(example_queries))

score: 0.07906002902973568


Now it's up to you to figure out if and how it's possible to make the search engine perform better on these queries. How would you query a search engine if you wanted to know about these topics? Experiment a bit.

**Task 2**: Try to manually come up with ways to rewrite or reformulate the queries so the performance improves.

**Important**: Make sure that the query IDs match! Otherwise, evaluation will not work.


In [5]:
example_queries_rewritten = pd.DataFrame(
    [
        # Write reflection about method used
        [
            "443848",
            # Original: does anybody know where i could get a free guide on how to train a siberian husky
            # Adding more general terms such as "Dog" helps find more relevant results
            "siberian husky dog train",
            #"siberian husky train dog",
        ],
        [
            "1783010",
            # Original: blaphsemy
            # Fix typos
            "what is Blasphemy",
            #"what Blasphemy is",
        ],
        [
            "2838988",
            # Original: how can i get a cork out of not into a wine bottle without a corkscrew
            "cork bottle without corkscrew",
            #"bottle corkscrew cork without",
        ],
    ],
    columns=["qid", "query"],
)
print("score after rewriting:", evaluate(example_queries_rewritten))
print("idf:", get_tf_idf("blasphemy"))

14:43:40.266 [main] WARN org.terrier.querying.ApplyTermPipeline -- The index has no termpipelines configuration, and no control configuration is found. Defaulting to global termpipelines configuration of ''. Set a termpipelines control to remove this warning.
score after rewriting: 0.10694392789340007
idf: 4.4387635820305755e-05


In [6]:
!pip install pyspellchecker
!pip install nltk

# An automatic approach

In this last part, we'll try to come up with an automatic approach to perform query re-writing. Use your findings from task 2 for this.

**Task 3**: Implement a function that automatically re-writes any input query.

You can use any approach or library you want for this task. However, keep in mind that simple ideas often work well!


In [9]:
import numpy as np
from nltk import WordNetLemmatizer, word_tokenize
import re
from spellchecker import SpellChecker
import nltk
from nltk.corpus import stopwords, wordnet
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer, util

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
custom_stopwords = {'something', 'get'}
stop_words = set(stopwords.words('english')).union(custom_stopwords)
spell = SpellChecker()
keybert = KeyBERT()
lemmatizer = WordNetLemmatizer()
bert_model = SentenceTransformer("all-MiniLM-L6-v2")

# Amount of documents to use for query expansion
TOP_N = 5
# Amount of keywords to use for query expansion
TOP_KEYWORDS = 5

SIMILARITY_THRESHOLD = 0.7


def remove_typos(query: str) -> str:
    """
    Remove typos using spellchecker.
    """
    # Tokenize the query
    words = word_tokenize(query)

    # Lemmatize words
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]

    # Correct spelling
    corrected_words = [spell.correction(word) if spell.correction(word) else word for word in lemmatized_words]

    return " ".join(corrected_words)


def remove_stopwords(query: str) -> str:
    """
    Remove stop words from tokens.
    """
    tokens = word_tokenize(query)
    filtered = [token for token in tokens if not token in stop_words]
    filtered = filtered if len(filtered) > 0 else tokens
    return " ".join(filtered)


def expand_contractions(query: str) -> str:
    """Expands common contractions manually"""
    contractions = {
        "n't": " not", "'re": " are", "'s": " is", "'d": " would", "'ll": " will",
        "'t": " not", "'ve": " have", "'m": " am"
    }
    for contraction, full_form in contractions.items():
        query = query.replace(contraction, full_form)
    return query


def normalize(query: str) -> str:
    return " ".join(set(re.sub(r"[^a-zA-Z0-9']", '', token.lower()) for token in word_tokenize(expand_contractions(query))))


def bert_similarity(word: str, query: str) -> float:
    """Compute cosine similarity using BERT sentence embeddings"""
    word_embedding = bert_model.encode(word, convert_to_tensor=True)
    query_embedding = bert_model.encode(query, convert_to_tensor=True)
    similarity_score = util.pytorch_cos_sim(word_embedding, query_embedding).item()
    return similarity_score


def add_context(query: str) -> str:
    query = normalize(query)
    tokens = word_tokenize(query)

    # Get top N results with current query
    results = search(query)['text'][:TOP_N]
    # Find keywords in each document retrieved
    context_words = []
    for result in results:
        keywords = keybert.extract_keywords(result, seed_keywords=tokens)
        sorted_keywords = sorted(keywords, key=lambda x: x[1], reverse=True)[:TOP_KEYWORDS]
        for keyword, score in sorted_keywords:
            if bert_similarity(keyword, query) > SIMILARITY_THRESHOLD:
                context_words.append(keyword)

    expanded_tokens = context_words + tokens
    print('expanded tokens:', context_words)
    return " ".join(expanded_tokens)


def rewrite_query(query: str) -> str:
    print("orig query:", query)
    operations = [normalize, remove_typos, remove_stopwords, add_context, normalize]
    for operation in operations:
        query = operation(query)

    print("rewriting query:", query)
    return query


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/maxdegroot/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/maxdegroot/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/maxdegroot/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


This time, we'll evalute on _all_ queries in the dataset. This will give us a more general result:


In [10]:
print("score:", evaluate_all())

score: 0.06179994498738492


Are you able to improve the overall performance using your rewriting approach?


In [10]:
print("score after rewriting", evaluate_all(rewrite_query))

orig query: how can we get concentration onsomething
expanded tokens: []
rewriting query: concentration
orig query: why doesn t the water fall off earth if it s round
expanded tokens: []
rewriting query: water fall earth round
orig query: how do i determine the charge of the iron ion in fecl3
expanded tokens: []
rewriting query: charge feel ion iron determine
orig query: i have mice how do i get rid of them humanely
expanded tokens: []
rewriting query: mouse humanely rid
orig query: what does see leaflet mean on ept pregnancy test
expanded tokens: []
rewriting query: mean pregnancy test doe eat see leaflet
orig query: what is innate immunity
expanded tokens: []
rewriting query: innate immunity
orig query: how can i lose 30 pounds by june3
expanded tokens: []
rewriting query: 30 june pound lose
orig query: what are the words to write the sound of raindrops moving train scribbling w pencil on paper figuratively
expanded tokens: []
rewriting query: scribbling pencil sound write paper w tr

KeyboardInterrupt: 